#### varaibles
- real gdp per capita growth - trend + cycle of real values? diff og gdp real per capita
- inflation = inflation trend + measurement errors???
- short rate = inflation trend + real rate trend 
- long rate = inflation trend + real rate trend + term trend
- inflation expectations??
- growth expextations??
- 

In [2]:
import Pkg
Pkg.activate("../")

using Revise

includet("../src/TCVAR/TCVAR.jl")

  Activating project at `~/projects/MacroFinanceScenarios`


In [3]:
using .TCVAR
using DataFrames, XLSX, TimeSeries
using StatsBase
using LinearAlgebra
using Plots


In [6]:
pwd()

"/home/matsz/projects/MacroFinanceScenarios/analisys"

In [9]:
df = DataFrame(XLSX.readtable("../data/DelNegro.xlsx", "Sheet1"))
data_source = TimeArray(df; timestamp = :Date)

s   = timestamp(data_source[:BILL])
vals = values(data_source[:BILL])

# widen so the array can hold `missing`
new_vals = convert(Array{Union{Missing, eltype(vals)}}, vals)

new_vals[s .>= Date(2008, 10,1), :] .= missing

TBILL = TimeArray(s, new_vals, [:BILL])

data = merge(data_source[:Pi], data_source[:EPi], TBILL, data_source[:EBILL], data_source[:TBlong])


presample, data = data[Date(1954,01,01):Date(1959,12,31)], data[Date(1960,01,01):Date(2016,12,31)]

term = presample[:TBlong] .- presample[:BILL]
real_rate = presample[:BILL] .- presample[:Pi]

data =convert(Matrix{Union{Missing, Float64}},values(data))


display(presample)
display(data) 


#real_rate = presample[:TB3MS] .- presample[:GDPDEF]   =#

#presample = merge(presample[[:A939RX0Q048SBEA, :GDPDEF]], real_rate, term)
 


LoadError: TypeError: non-boolean (Missing) used in boolean context

In [5]:
show(err)

1-element ExceptionStack:
LoadError: XLSXError: File ../../../data/DelNegro.xlsx not found.
Stacktrace:
 [1] openxlsx(f::XLSX.var"#166#167"{Nothing, Nothing, Bool, Bool, Bool, Nothing, Bool, Bool, Nothing, String}, source::String; mode::String, enable_cache::Bool)
   @ XLSX ~/.julia/packages/XLSX/4zkI6/src/read.jl:466
 [2] openxlsx
   @ ~/.julia/packages/XLSX/4zkI6/src/read.jl:459 [inlined]
 [3] #readtable#164
   @ ~/.julia/packages/XLSX/4zkI6/src/read.jl:1301 [inlined]
 [4] readtable(source::String, sheet::String)
   @ XLSX ~/.julia/packages/XLSX/4zkI6/src/read.jl:1289
 [5] top-level scope
   @ In[4]:1
 [6] eval(m::Module, e::Any)
   @ Core ./boot.jl:489
in expression starting at In[4]:1

In [4]:
var.([values(presample[:Pi]), values(term), values(real_rate)])

3-element Vector{Float64}:
 2.2741804700492754
 0.4066775362318842
 1.595023391788406

In [5]:
mean.([values(presample[:Pi]), values(term), values(real_rate)])

3-element Vector{Float64}:
 1.706988333333333
 1.0258333333333336
 0.5717616666666668

In [6]:
presample_mean = mean(presample)
presample_mean = round.(presample_mean, digits=2)
display("presample mean")
display(presample_mean)

presample_variance = var(presample)
presample_variance = round.(presample_variance, digits=2)
display("presample variance")
display(presample_variance)
display(presample_variance .^ .5) 

display(var(term))

"presample mean"

1×5 TimeArray{Missing, 2, Date, Matrix{Missing}} 1959-10-01 to 1959-10-01
┌────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│            │ Pi      │ EPi     │ EBILL   │ TBlong  │ BILL    │
├────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ 1959-10-01 │ missing │ missing │ missing │ missing │ missing │
└────────────┴─────────┴─────────┴─────────┴─────────┴─────────┘

"presample variance"

1×5 TimeArray{Missing, 2, Date, Matrix{Missing}} 1959-10-01 to 1959-10-01
┌────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│            │ Pi      │ EPi     │ EBILL   │ TBlong  │ BILL    │
├────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ 1959-10-01 │ missing │ missing │ missing │ missing │ missing │
└────────────┴─────────┴─────────┴─────────┴─────────┴─────────┘

1×5 TimeArray{Missing, 2, Date, Matrix{Missing}} 1959-10-01 to 1959-10-01
┌────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│            │ Pi      │ EPi     │ EBILL   │ TBlong  │ BILL    │
├────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ 1959-10-01 │ missing │ missing │ missing │ missing │ missing │
└────────────┴─────────┴─────────┴─────────┴─────────┴─────────┘

1×1 TimeArray{Float64, 1, Date, Vector{Float64}} 1959-10-01 to 1959-10-01
┌────────────┬─────────────┐
│            │ TBlong_BILL │
├────────────┼─────────────┤
│ 1959-10-01 │    0.406678 │
└────────────┴─────────────┘

In [7]:
n = 5 #number of observatin variables
nt = 3 #number of trends

priors = (
        initial_trend_mean = [2., .5, 1.],
        initial_cycle_mean = zeros(n),
        initial_trend_covariance = diagm(fill(1,nt)),
        trend_covariance_df = 100,
        trend_covariance_mean = diagm([2., 1., 1.] ./ 400),
        cycle_coeff_mean = zeros(n, n),
        cycle_coeff_shrinkage_param = .2,
        cycle_covariance_mean = diagm([2., .5, .5, 1., 1.]),  
        cycle_covariance_df = n+2
        )


(initial_trend_mean = [2.0, 0.5, 1.0], initial_cycle_mean = [0.0, 0.0, 0.0, 0.0, 0.0], initial_trend_covariance = [1 0 0; 0 1 0; 0 0 1], trend_covariance_df = 100, trend_covariance_mean = [0.005 0.0 0.0; 0.0 0.0025 0.0; 0.0 0.0 0.0025], cycle_coeff_mean = [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], cycle_coeff_shrinkage_param = 0.2, cycle_covariance_mean = [2.0 0.0 … 0.0 0.0; 0.0 0.5 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 1.0], cycle_covariance_df = 7)

In [8]:
priors.cycle_covariance_mean

5×5 Matrix{Float64}:
 2.0  0.0  0.0  0.0  0.0
 0.0  0.5  0.0  0.0  0.0
 0.0  0.0  0.5  0.0  0.0
 0.0  0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.0  1.0

In [41]:
observation_trend_mapping  = [1 0 0  
                             1 0 0  
                             1 1 0  
                             1 1 1  
                             1 1 0 ]



trend_states_samples, cycle_states_samples, trend_covariance_samples, betas_samples, sigmas_samples = TCVAR.gibs_sampler(data, observation_trend_mapping, priors; burnin = 50_000, n_samples = 50_000, thin=25)

trend_states_mean, trend_states_lower, trend_states_upper = TCVAR.compute_posterior_statistics(trend_states_samples, credible_level=0.68)  
cycle_states_mean, cycle_states_lower, cycle_states_upper = TCVAR.compute_posterior_statistics(cycle_states_samples, credible_level=0.95) 

String: "not posistive define [2.1717183112457326 -0.08324558966625892 -0.10048622093902411 -5.483732506322858e16 -7.695338347439446e17 -1.3915073738576952e17 -5.435216838562355e17 -4.004768322006923e16; -0.08324558966625892 1.6641234552321367 0.060129156307207415 2.7060168770450428e16 3.797361636209061e17 6.8665684072844616e16 2.6820762096967165e17 1.976203371610647e16; -0.10048622093902411 0.060129156307207415 1.8538015734833648 9.05373027085947e17 1.270512696605553e19 2.2974009797694177e18 8.973630125692283e18 6.611936695111826e17; -5.483732506322858e16 2.7060168770450428e16 9.05373027085947e17 1.261392871335469e104 1.7701164166528406e105 3.200807989395997e104 1.2502330787324567e105 9.211948626059273e103; -7.695338347439446e17 3.797361636209061e17 1.270512696605553e19 1.7701164166528406e105 2.4840128894131865e106 4.491709261264849e105 1.7544580984196558e106 1.2927171856308525e105; -1.3915073738576952e17 6.8665684072844616e16 2.2974009797694177e18 3.200807989395997e104 4.491709261264

In [10]:
ismissing(data[1,3])

true

In [11]:
priors.initial_trend_covariance

3×3 Matrix{Int64}:
 1  0  0
 0  1  0
 0  0  1

In [12]:
n_variables, n_trends = size(observation_trend_mapping)
initial_state_covariance = [priors.initial_trend_covariance zeros(n_trends, n_variables)
                                 zeros(n_variables, n_trends) priors.cycle_covariance_mean]

8×8 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  2.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.5  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.5  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0

In [13]:
st = 1
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
st = 2
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
st = 3
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
st = 4
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
st=1
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
st=2
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [19]:
st=3
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [20]:
st=4
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [21]:
summarystats(trend_covariance_samples)

UndefVarError: UndefVarError: `trend_covariance_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [22]:
summarystats(betas_samples)

UndefVarError: UndefVarError: `betas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [23]:
summarystats(sigmas_samples)

UndefVarError: UndefVarError: `sigmas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [24]:
plot(betas_samples)

UndefVarError: UndefVarError: `betas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [25]:
Σc = mean(sigmas_samples).nt.mean
Σc = reshape(Σc, n, n)
display(Σc)

β = mean(betas_samples).nt.mean
β = reshape(β, n, n*1)
display(β)

Στ = mean(trend_covariance_samples).nt.mean
Στ = reshape(Στ, 4, 4)
display(cov2cor(Στ))
display(diag(Στ) .^ .5)

UndefVarError: UndefVarError: `sigmas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [26]:
display(diag(Σc) .^ .5)

display(diag(Στ) .^ .5)

UndefVarError: UndefVarError: `Σc` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [27]:
model = tc_var(observation_tend_mapping, β, Στ, Σc, priors.initial_trend_mean, priors.initial_cycle_mean, priors.initial_trend_covariance, priors.cycle_covariance_mean)

initial_states = [trend_states_mean[end,:]; cycle_states_mean[end,:]]

n_samples = 2_000
T = 100
states = zeros(n_samples, T, 8)

observations = zeros(n_samples, T, n)

for s in 1:2_000
    states[s, :, :], observations[s, :, :] = sample(model, initial_states, T)
end


UndefVarError: UndefVarError: `observation_tend_mapping` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [28]:
transformed_scenarios = permutedims(observations, (3, 2, 1))[[1,2],:,:] ./400
periods = [1, 5, 10, 25]
freq = 4
assets_names = ["GDP", "CPI"]
ret_in_years = cum_returns_in_periods(transformed_scenarios, periods, freq, true)
print_scenarios_summary(ret_in_years, assets_names, string.(periods))

for a in 1:2
    print_scenarios_percentiles(ret_in_years[a, :, :], [.01, 0.025, .05, .25, .5, .75, .95, .975, .99], string.(periods), string.(assets_names[a]))
end  

UndefVarError: UndefVarError: `observations` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [29]:
transformed_scenarios = permutedims(observations[:,:,[3,4]] , (3, 2, 1))
freq = 4

transformed_scenarios = transformed_scenarios[:,freq:freq:end,:]

periods = [1, 5, 10, 25]


assets_names = ["ShortRate", "LongRate"]
ret_in_years = transformed_scenarios = transformed_scenarios[:,periods,:]
print_scenarios_summary(ret_in_years, assets_names, string.(periods))

for a in 1:2
    print_scenarios_percentiles(ret_in_years[a, :, :], [.01, 0.025, .05, .25, .5, .75, .95, .975, .99], string.(periods), string.(assets_names[a]))
end  

UndefVarError: UndefVarError: `observations` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [30]:
n_observations = size(model.T,1)
n_trends = 4
n_time_steps = size(data,1)
n_draws = 1000
trends_states = zeros(n_draws, n_time_steps, n_trends)

for s in 1:n_draws
    state_smoothed_samples = carter_kohn_sampler(model, values(data))
    trends_states[s,:,:] = state_smoothed_samples[:, 1:n_trends]
end

trend_states_mean, trend_states_lower, trend_states_upper = TCVAR.compute_posterior_statistics(trend_states_samples, credible_level=0.95) 

UndefVarError: UndefVarError: `model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [31]:
st = 2
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [32]:
st = 3
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.